In [12]:
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])?  y


In [13]:
# 모듈 import
import os
import glob # 조건에 맞는 파일명을 찾아서 싹 가져오는 함(이미지 찾기)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.utils import plot_model

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB4
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.layers import Concatenate, Dropout
from tensorflow.keras.layers import BatchNormalization, Activation
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import SGD, RMSprop, Adam
from tensorflow.keras import mixed_precision
mixed_precision.set_global_policy('mixed_float16')

In [14]:
# 전체 이미지 파일 load
base_path = "/home/govlept1004/jupyter_home/data/cat_dog/train"

# cat과 dog 이미지 경로 가져오기
cat_images = glob.glob(os.path.join(base_path, "cat.*.jpg"))
dog_images = glob.glob(os.path.join(base_path, "dog.*.jpg"))

In [15]:
# 데이터 분리
# image_paths = cat_images + dog_images
# labels = [0] * len(cat_images) + [1] * len(dog_images)

x_data = []
t_data = []

for label, category in enumerate(['cat', 'dog']):  # cat → 0, dog → 1
    paths = glob.glob(f"/home/govlept1004/jupyter_home/data/cat_dog/train/{category}.*.jpg")
    x_data.extend(paths)
    t_data.extend([label] * len(paths))

In [16]:
# 학습용과 테스트용으로 분리 (총 3개 분리)
x_data_train, x_data_test, t_data_train, t_data_test = train_test_split(x_data,
                                                                       t_data,
                                                                       test_size=0.2,
                                                                       stratify=t_data)
x_data_train, x_data_valid, t_data_train, t_data_valid = train_test_split(x_data_train,
                                                                          t_data_train,
                                                                          test_size=0.2,
                                                                          stratify=t_data_train)

In [17]:
# Dataset 생성

BATCH_SIZE = 64
IMAGE_SIZE = 380

# 이미지 불러오고 전처리까지 하는 함수
def parse_image(filename, label):
    image = tf.io.read_file(filename)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [IMAGE_SIZE,IMAGE_SIZE])
    image = tf.keras.applications.efficientnet.preprocess_input(image)
    return image, label

# 학습 데이터만 증강하는 함수
def augment(image, label):
    image = tf.image.random_flip_left_right(image)
    return image, label
    
# 데이터셋 생성 함수
def make_dataset(x,t,train=False):
    dataset = tf.data.Dataset.from_tensor_slices((x,t))
    dataset = dataset.map(parse_image,
                          num_parallel_calls=tf.data.AUTOTUNE)
    if train:
        dataset = dataset.shuffle(buffer_size=1000)
        dataset = dataset.map(augment,
                              num_parallel_calls=tf.data.AUTOTUNE)    
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [18]:
train_dataset = make_dataset(x_data_train, t_data_train, train=True)
validation_dataset = make_dataset(x_data_valid, t_data_valid, train=False)
test_dataset = make_dataset(x_data_test, t_data_test, train=False)

In [19]:
# model
model_base = EfficientNetB4(weights='imagenet',
                            include_top=False,
                            input_shape=(IMAGE_SIZE,IMAGE_SIZE,3))
for layer in model_base.layers:
    layer.trainable = False

In [20]:
model = Sequential()
model.add(model_base)
model.add(GlobalAveragePooling2D())
model.add(Dense(units=64))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(rate=0.3))
model.add(Dense(units=1,
               activation='sigmoid'))

In [21]:
# model 설정
model.compile(optimizer=Adam(learning_rate=1e-4),
             loss='binary_crossentropy',
             metrics=['accuracy'])

In [22]:
es_callback = EarlyStopping(monitor='val_loss',
                           patience=5,
                           restore_best_weights=True,
                           verbose=1)
cp_callback = ModelCheckpoint(filepath='./efficientnetb4_weights.h5',
                             save_best_only=True,
                             save_weights_only=True,
                             monitor='val_accuracy',
                             verbose=1)

In [23]:
# 1차 학습 진행
model.fit(train_dataset,
         epochs=30,          
         validation_data=validation_dataset,
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/30
250/250 [==============================] - ETA: 0s - loss: 0.1174 - accuracy: 0.9706   
Epoch 1: val_accuracy improved from -inf to 0.99625, saving model to ./efficientnetb4_weights.h5
250/250 [==============================] - 99s 362ms/step - loss: 0.1174 - accuracy: 0.9706 - val_loss: 0.0493 - val_accuracy: 0.9962
Epoch 2/30
250/250 [==============================] - ETA: 0s - loss: 0.0396 - accuracy: 0.9956  
Epoch 2: val_accuracy improved from 0.99625 to 0.99700, saving model to ./efficientnetb4_weights.h5
250/250 [==============================] - 89s 354ms/step - loss: 0.0396 - accuracy: 0.9956 - val_loss: 0.0236 - val_accuracy: 0.9970
Epoch 3/30
250/250 [==============================] - ETA: 0s - loss: 0.0276 - accuracy: 0.9966  
Epoch 3: val_accuracy improved from 0.99700 to 0.99725, saving model to ./efficientnetb4_weights.h5
250/250 [==============================] - 89s 356ms/step - loss: 0.0276 - accuracy: 0.9966 - val_loss: 0.0186 - val_accuracy: 0.9973
Epoch 

In [24]:
# Fine Tuning
model_base.trainable = True

for layer in model_base.layers[:-30]:
    layer.trainable = False

In [25]:
# model 재설정
model.compile(optimizer=Adam(learning_rate=1e-5),
             loss='binary_crossentropy',
             metrics=['accuracy'])

In [27]:
# model 재학습
model.fit(train_dataset,
         epochs=25,          
         validation_data=test_dataset,
         callbacks=[es_callback, cp_callback],
         verbose=1)

Epoch 1/25
250/250 [==============================] - ETA: 0s - loss: 0.0286 - accuracy: 0.9923   
Epoch 1: val_accuracy did not improve from 0.99750
250/250 [==============================] - 111s 395ms/step - loss: 0.0286 - accuracy: 0.9923 - val_loss: 0.0146 - val_accuracy: 0.9962
Epoch 2/25
250/250 [==============================] - ETA: 0s - loss: 0.0165 - accuracy: 0.9964  
Epoch 2: val_accuracy did not improve from 0.99750
250/250 [==============================] - 98s 390ms/step - loss: 0.0165 - accuracy: 0.9964 - val_loss: 0.0138 - val_accuracy: 0.9967
Epoch 3/25
250/250 [==============================] - ETA: 0s - loss: 0.0150 - accuracy: 0.9966  
Epoch 3: val_accuracy did not improve from 0.99750
250/250 [==============================] - 98s 392ms/step - loss: 0.0150 - accuracy: 0.9966 - val_loss: 0.0128 - val_accuracy: 0.9967
Epoch 4/25
250/250 [==============================] - ETA: 0s - loss: 0.0121 - accuracy: 0.9973  
Epoch 4: val_accuracy did not improve from 0.99750


In [30]:
# 예측 수행
predict = model.evaluate(test_dataset, verbose=1)
print(predict)

79/79 [==============================] - 21s 265ms/step - loss: 0.0170 - accuracy: 0.9950
[0.01701933704316616, 0.9950000047683716]
